<a href="https://www.kaggle.com/code/awedatk/snn-codenew?scriptVersionId=314230411" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# =========================================================
#  Spiking Autoencoder Cortex Circuit (SACC) – STFT Version
#  Simplified for IEEE EIT 2026
# =========================================================

!pip install brian2 torchmetrics librosa soundfile -q

import numpy as np
import librosa, soundfile as sf
import matplotlib.pyplot as plt
from brian2 import *
from scipy.ndimage import gaussian_filter1d
from torchmetrics.audio import ScaleInvariantSignalNoiseRatio
import torch

# ---------- 1. Load and preprocess audio ----------
clean, sr = librosa.load('/kaggle/input/datasets/muhmagdy/valentini-noisy/clean_trainset_28spk_wav/p226_001.wav', sr=16000)
noisy, _  = librosa.load('/kaggle/input/datasets/muhmagdy/valentini-noisy/noisy_trainset_28spk_wav/p226_001.wav', sr=16000)

# Downsample for speed
sr_new = 4000
clean = librosa.resample(y=clean, orig_sr=sr, target_sr=sr_new)
noisy = librosa.resample(y=noisy, orig_sr=sr, target_sr=sr_new)

# Use first 0.5 seconds
segment_dur = 0.5
segment_len = int(sr_new * segment_dur)
clean = clean[:segment_len]
noisy = noisy[:segment_len]

# ---------- 2. STFT-based encoding ----------
# ---------- 2. STFT-based encoding ----------
def encode_stft_spikes(audio, sr=4000, num_channels=40, f_min=100, f_max=2000):
    """
    Divide signal into frequency bands using STFT magnitude.
    Each band becomes one input channel with firing rate ∝ energy.
    """
    n_fft = 256
    hop_length = n_fft // 4
    D = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
    magnitude = np.abs(D)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

    bands = np.linspace(f_min, f_max, num_channels+1)
    rates = np.zeros(num_channels)
    for i in range(num_channels):
        idx = np.where((freqs >= bands[i]) & (freqs < bands[i+1]))[0]
        if len(idx) > 0:
            band_energy = np.mean(magnitude[idx, :])
           # rates[i] = np.clip(band_energy * 100, 0, 300)  # Increased scaling
            rates[i] = np.clip(band_energy * 400, 0, 500)  # boost STFT scaling

    return rates * Hz


num_inputs = 40
spike_rates = encode_stft_spikes(noisy, sr_new, num_channels=num_inputs)

plt.figure(figsize=(8,3))
plt.bar(np.arange(num_inputs), spike_rates/Hz)
plt.xlabel("Input neuron index (frequency band)")
plt.ylabel("Firing rate (Hz)")
plt.title("STFT-based Spike Encoding (frequency bands)")
plt.show()

# ---------- 3. Brian2 SACC Network ----------
start_scope()
duration = segment_dur * second
num_outputs = num_inputs
eta = 0.02

input_layer = PoissonGroup(num_inputs, rates=spike_rates)
output_layer = NeuronGroup(num_outputs, '''
    dv/dt = -v/(10*ms) : 1
    ''', threshold='v>1', reset='v=0', method='exact')

# STDP Synapses
S = Synapses(input_layer, output_layer, '''
    w : 1
    dpre/dt = -pre/(20*ms) : 1 (event-driven)
    dpost/dt = -post/(20*ms) : 1 (event-driven)
    ''',
    on_pre='''
    v_post += w
    pre += 1.
    w = clip(w + eta * post, 0, 1)
    ''',
    on_post='''
    post += 1.
    w = clip(w + eta * pre, 0, 1)
    ''')
S.connect(p=0.3)
S.w = '0.5*rand()'

# Monitors
M_out = SpikeMonitor(output_layer)
M_w = StateMonitor(S, 'w', record=range(5))

# Run simulation
run(duration)
print("STFT-based STDP simulation completed.")

# ---------- 4. Decode spikes ----------
spike_counts = np.zeros(num_outputs)
for i in M_out.i:
    spike_counts[i] += 1

decoded = gaussian_filter1d(spike_counts, sigma=2)
decoded = decoded / (np.max(decoded) + 1e-8)
decoded_full = np.interp(np.arange(len(noisy)),
                         np.linspace(0, len(noisy)-1, num_outputs),
                         decoded)

sf.write("sacc_stft_output.wav", decoded_full, sr_new)
print("Saved: sacc_stft_output.wav")

# ---------- 5. Evaluate ----------
metric = ScaleInvariantSignalNoiseRatio()
clean_t = torch.tensor(clean)
denoised_t = torch.tensor(decoded_full[:len(clean)])
si_snr_value = metric(denoised_t, clean_t)
print("SACC (STFT-trained) SI-SNR:", si_snr_value.item(), "dB")

energy_proxy = np.sum(spike_counts)
print("Total spikes (energy proxy):", energy_proxy)

# ---------- 6. Visualizations ----------
plt.figure(figsize=(10,4))
plt.plot(M_out.t/ms, M_out.i, '.k', markersize=1)
plt.xlabel('Time (ms)')
plt.ylabel('Neuron index')
plt.title('Output Spike Raster (STFT input)')
plt.show()

plt.figure(figsize=(10,3))
for i in range(5):
    plt.plot(M_w.t/ms, M_w.w[i], label=f'w{i}')
plt.xlabel('Time (ms)')
plt.ylabel('Weight value')
plt.title('Evolution of first 5 synaptic weights')
plt.legend()
plt.show()


In [ ]:
'''

this is new version with adaptive weights
'''
from brian2 import *
import numpy as np
import librosa
import os
import matplotlib.pyplot as plt

train_clean = "/kaggle/input/datasets/muhmagdy/valentini-noisy/clean_trainset_28spk_wav"

sr = 16000
n_fft = 512
hop_length = 128
n_mels = 80

receptor_dim = 128
kc_dim = 2000
conn_per_kc = 6

pn_rate_scale = 150.0
frame_dt = hop_length / sr

w_min = 0.0
w_max = 1.5
A_plus = 0.01
A_minus = -0.012
taupre_val = 20 * ms
taupost_val = 20 * ms

defaultclock.dt = 0.5 * ms
np.random.seed(42)

files = sorted([os.path.join(train_clean, f) for f in os.listdir(train_clean) if f.endswith(".wav")])
audio_path = files[0]

audio, _ = librosa.load(audio_path, sr=sr)
audio = audio[: sr * 3]

mel = librosa.feature.melspectrogram(
    y=audio,
    sr=sr,
    n_fft=n_fft,
    hop_length=hop_length,
    n_mels=n_mels,
    power=1.0
)

logmel = np.log1p(mel).T
delta = librosa.feature.delta(logmel.T).T
delta2 = librosa.feature.delta(logmel.T, order=2).T

x = np.concatenate([logmel, delta, delta2], axis=1)
x = (x - x.mean(axis=0, keepdims=True)) / (x.std(axis=0, keepdims=True) + 1e-6)

W_receptor = np.random.randn(x.shape[1], receptor_dim)
r = np.maximum(0.0, x @ W_receptor)

z = r / (1e-4 + 0.1 + 0.5 * r.sum(axis=1, keepdims=True))
z = z / (z.max() + 1e-8)

rates = np.clip(z * pn_rate_scale, 0.0, pn_rate_scale)
rate_ta = TimedArray(rates * Hz, dt=frame_dt * second)

start_scope()

PN = PoissonGroup(receptor_dim, rates='rate_ta(t, i)')

KC = NeuronGroup(
    kc_dim,
    '''
    dv/dt = (
        -(v - v_rest)
        + Delta_T*exp((v - v_th)/Delta_T)
        - w_adapt
        + g_exc*(E_exc - v)
        + g_inh*(E_inh - v)
    ) / tau_m : 1

    dw_adapt/dt = (a*(v - v_rest) - w_adapt) / tau_w : 1
    dg_exc/dt = -g_exc / tau_exc : 1
    dg_inh/dt = -g_inh / tau_inh : 1

    v_rest : 1
    v_reset : 1
    v_th : 1
    v_spike : 1
    Delta_T : 1
    a : 1
    b : 1
    E_exc : 1
    E_inh : 1
    tau_m : second
    tau_w : second
    tau_exc : second
    tau_inh : second
    ''',
    threshold='v > v_spike',
    reset='''
    v = v_reset
    w_adapt += b
    ''',
    method='euler'
)

KC.v = 0.0
KC.v_rest = 0.0
KC.v_reset = 0.0
KC.v_th = 1.0
KC.v_spike = 2.0
KC.Delta_T = 0.2
KC.a = 0.01
KC.b = 0.05
KC.E_exc = 2.0
KC.E_inh = -1.0
KC.tau_m = 20 * ms
KC.tau_w = 100 * ms
KC.tau_exc = 5 * ms
KC.tau_inh = 10 * ms

APL = NeuronGroup(
    1,
    '''
    dv/dt = (
        -(v - v_rest)
        + Delta_T*exp((v - v_th)/Delta_T)
        + g_exc*(E_exc - v)
    ) / tau_m : 1

    dg_exc/dt = -g_exc / tau_exc : 1

    v_rest : 1
    v_reset : 1
    v_th : 1
    v_spike : 1
    Delta_T : 1
    E_exc : 1
    tau_m : second
    tau_exc : second
    ''',
    threshold='v > v_spike',
    reset='v = v_reset',
    method='euler'
)

APL.v = 0.0
APL.v_rest = 0.0
APL.v_reset = 0.0
APL.v_th = 1.0
APL.v_spike = 2.0
APL.Delta_T = 0.2
APL.E_exc = 2.0
APL.tau_m = 10 * ms
APL.tau_exc = 5 * ms

src = []
dst = []
wts = []

for j in range(kc_dim):
    idx = np.random.choice(receptor_dim, size=conn_per_kc, replace=False)
    for i in idx:
        src.append(int(i))
        dst.append(int(j))
        wts.append(float(np.random.uniform(0.3, 1.0)))

src = np.array(src, dtype=int)
dst = np.array(dst, dtype=int)
wts = np.array(wts, dtype=float)

print("len(src):", len(src))
print("len(dst):", len(dst))
print("len(wts):", len(wts))

w_min = 0.0
w_max = 1.5
A_pre = 0.003
A_post = 0.004
taupre_val = 20 * ms
taupost_val = 20 * ms

S1 = Synapses(
    PN, KC,
    '''
    w : 1
    dpretrace/dt = -pretrace / taupre : 1 (event-driven)
    dposttrace/dt = -posttrace / taupost : 1 (event-driven)
    taupre : second (constant)
    taupost : second (constant)
    ''',
    on_pre='''
    g_exc_post += w
    pretrace += A_pre
    w = clip(w - posttrace, w_min, w_max)
    ''',
    on_post='''
    posttrace += A_post
    w = clip(w + 0.25*pretrace, w_min, w_max)
    '''
)

S1.connect(i=src, j=dst)
S1.w = wts
S1.taupre = taupre_val
S1.taupost = taupost_val

initial_weights = np.array(S1.w[:])

S2 = Synapses(KC, APL, on_pre='g_exc_post += 0.03')
S2.connect()

S3 = Synapses(APL, KC, on_pre='g_inh_post += 0.15')
S3.connect()

M_kc = SpikeMonitor(KC)
M_apl = SpikeMonitor(APL)

run(len(rates) * frame_dt * second)

final_weights = np.array(S1.w[:])

spike_counts = np.zeros((len(rates), kc_dim), dtype=np.int32)
times = np.array(M_kc.t / second)
neurons = np.array(M_kc.i)
frames = np.floor(times / frame_dt).astype(int)

for f, i in zip(frames, neurons):
    if 0 <= f < len(rates):
        spike_counts[f, i] += 1

sparse = (spike_counts > 0).astype(np.int8)
active = sparse.sum(axis=1)

apl_times = np.array(M_apl.t / second)
apl_frames = np.floor(apl_times / frame_dt).astype(int)
apl_counts = np.zeros(len(rates), dtype=np.int32)

for f in apl_frames:
    if 0 <= f < len(rates):
        apl_counts[f] += 1

print("audio_path:", audio_path)
print("frames:", len(rates))
print("KC total spikes:", len(M_kc.i))
print("APL total spikes:", len(M_apl.i))
print("Average active KCs per frame:", active.mean())
print("Min active KCs:", active.min())
print("Max active KCs:", active.max())
print("Sparsity ratio:", sparse.mean())
print("Initial mean weight:", initial_weights.mean())
print("Final mean weight:", final_weights.mean())
print("Mean abs weight change:", np.mean(np.abs(final_weights - initial_weights)))
print("Fraction near max weight:", np.mean(final_weights > 0.95 * w_max))
print("Fraction near min weight:", np.mean(final_weights < 0.05 * w_max))

plt.figure(figsize=(12, 3))
plt.plot(audio[:sr])
plt.title("Audio Signal")
plt.xlabel("Sample")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.imshow(logmel.T, aspect='auto', origin='lower')
plt.title("Log-Mel Spectrogram")
plt.xlabel("Frame")
plt.ylabel("Mel Bin")
plt.colorbar()
plt.tight_layout()
plt.show()

spike_t, spike_k = np.where(sparse > 0)
plt.figure(figsize=(12, 5))
plt.scatter(spike_t, spike_k, s=2)
plt.title("KC Spike Raster")
plt.xlabel("Frame")
plt.ylabel("KC Index")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 3))
plt.plot(active)
plt.title("Active KCs per Frame")
plt.xlabel("Frame")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 3))
plt.plot(apl_counts)
plt.title("APL Spikes per Frame")
plt.xlabel("Frame")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(active, bins=20)
plt.title("Sparsity Distribution")
plt.xlabel("Active KCs")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(initial_weights, bins=30)
plt.title("Initial PN→KC Weights")
plt.xlabel("Weight")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
plt.hist(final_weights, bins=30)
plt.title("Final PN→KC Weights")
plt.xlabel("Weight")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


plt.figure(figsize=(6, 4))
plt.hist(final_weights - initial_weights, bins=30)
plt.title("Weight Change Distribution")
plt.xlabel("Δ weight")
plt.ylabel("Count")
plt.tight_layout()
plt.show()